# NFT Earnings — расчёт заработка кошельков (Google Colab)

Считает **realized profit** по кошелькам NFT-коллекции на Robinhood-чейне через Alchemy.

Модель: `realized = total_sales - total_invested` (цены сделок берутся как нетто-дельта ETH+ERC20 по кошельку в каждой транзакции — маркетплейс-агностик).

**Что подготовить:** Alchemy-URL для robinhood-mainnet (вида `https://robinhood-mainnet.g.alchemy.com/v2/<KEY>`).

Порядок: Runtime → Run all. Ключ вводится скрыто.

## 1. Код (клонируем репозиторий)

In [ ]:
import os
if not os.path.isdir('nft'):
    !git clone -q https://github.com/c8539-coder/main.git nft
%cd nft
!git pull -q
print('OK, скрипт:', os.path.exists('tools/nft_earnings.py'))

## 2. Ключ Alchemy (robinhood-mainnet)

In [ ]:
import getpass
RH_RPC = getpass.getpass('RH_RPC (https://robinhood-mainnet.g.alchemy.com/v2/<KEY>): ').strip()
CONTRACT = (input('CONTRACT [0xae42d5511886590538160a3cbdb91388cf1e76a3]: ').strip()
            or '0xae42d5511886590538160a3cbdb91388cf1e76a3')
os.environ['RH_RPC'] = RH_RPC
os.environ['CONTRACT'] = CONTRACT
print('Готово. chainId проверим ниже.')

## 3. Сверка по одному кошельку

Сравни цифры с эталонным скриншотом (`0xb180…d8a8`): BOUGHT/SOLD/HOLDING, TOTAL INVESTED/SALES, REALIZED PROFIT.

In [ ]:
WALLET = (input('WALLET [0xb180e3fde77c0d4499a934014f437e8d442fd8a8]: ').strip()
          or '0xb180e3fde77c0d4499a934014f437e8d442fd8a8')
!python3 tools/nft_earnings.py --rpc "$RH_RPC" --contract "$CONTRACT" --wallet "$WALLET"
print('\n--- то же самое, но с вычетом газа (--gas) ---')
!python3 tools/nft_earnings.py --rpc "$RH_RPC" --contract "$CONTRACT" --wallet "$WALLET" --gas

## 4. Вся коллекция → CSV

Считает всех держателей коллекции и сохраняет `earnings.csv` (сортировка по заработку). Может занять время — зависит от числа кошельков и транзакций.

In [ ]:
!python3 tools/nft_earnings.py --rpc "$RH_RPC" --contract "$CONTRACT" --collection --out earnings.csv
import pandas as pd
df = pd.read_csv('earnings.csv')
print('Кошельков:', len(df))
df.head(25)

## 5. Скачать CSV

In [ ]:
from google.colab import files
files.download('earnings.csv')